
# FabricOps AI Data Quality — Widget Approval + Metadata Flow

This notebook reflects the intended DQ workflow:

1. Source data
2. Profile data
3. AI suggests candidate DQ rules
4. Human review and approval with widgets
5. Store approved DQ rules into a metadata table
6. Load approved DQ rules back from the metadata table
7. Pipeline applies the approved DQ rules deterministically
8. Split accepted vs quarantined rows

**Important boundary**
- AI-suggestable DQ rules: `not_null`, `unique_key`, `accepted_values`, `value_range`, `regex_format`


In [1]:
from fabricops_kit import (
    profile_for_dq,
    suggest_dq_rules,
    extract_dq_rules,
    review_dq_rules,
    build_dq_rule_history,
    load_active_dq_rules,
    run_dq_rules,
    split_dq_rows,
    assert_dq_passed,
    review_dq_rule_deactivations,
    build_dq_rule_deactivations,
)

SMOKE_DQ_CALLABLES = [
    profile_for_dq,
    suggest_dq_rules,
    extract_dq_rules,
    review_dq_rules,
    build_dq_rule_history,
    load_active_dq_rules,
    run_dq_rules,
    split_dq_rows,
    assert_dq_passed,
    review_dq_rule_deactivations,
    build_dq_rule_deactivations,
]
print('DQ smoke-check OK:', [fn.__name__ for fn in SMOKE_DQ_CALLABLES])


StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 3, Finished, Available, Finished, False)

_Framework DQ helper implementation moved to `fabricops_kit` imports above._


_Framework DQ helper implementation moved to `fabricops_kit` imports above._


_Framework DQ helper implementation moved to `fabricops_kit` imports above._


_Framework DQ helper implementation moved to `fabricops_kit` imports above._


_Framework DQ helper implementation moved to `fabricops_kit` imports above._


_Framework DQ helper implementation moved to `fabricops_kit` imports above._


_Framework DQ helper implementation moved to `fabricops_kit` imports above._


_Framework DQ helper implementation moved to `fabricops_kit` imports above._


In [9]:
import pandas as pd
import os

sample_dir = "/lakehouse/default/Files/fabricops_dq_demo"
sample_csv_path = f"{sample_dir}/email_logs.csv"

os.makedirs(sample_dir, exist_ok=True)

sample_pdf = pd.DataFrame(
    [
        {
            "message_id": "m001",
            "status": "Delivered",
            "sender_email": "alice@nus.edu.sg",
            "event_count": 1,
        },
        {
            "message_id": "m002",
            "status": "Resolved",
            "sender_email": "bob@nus.edu.sg",
            "event_count": 2,
        },
        {
            "message_id": None,
            "status": "Failed",
            "sender_email": "bad-email",
            "event_count": -1,
        },
        {
            "message_id": "m002",
            "status": "Delivered",
            "sender_email": "carol@nus.edu.sg",
            "event_count": 1,
        },
    ]
)

sample_pdf.to_csv(sample_csv_path, index=False)

print(sample_csv_path)

StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 11, Finished, Available, Finished, False)

/lakehouse/default/Files/fabricops_dq_demo/email_logs.csv


In [10]:
df_test = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/fabricops_dq_demo/email_logs.csv")
)

display(df_test)

StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 72d6ff26-32ac-4b77-ba95-7059712bce25)

In [11]:

TABLE_NAME = "EMAIL_LOGS"
BUSINESS_CONTEXT = """
Campus email metadata transformed into an analytics-ready email log table.
Each row represents one email message or event record.
MESSAGE_ID should uniquely identify the business grain.
STATUS should follow approved operational values.
"""

df_test = spark.createDataFrame(
    [
        {
            "message_id": "m001",
            "status": "Delivered",
            "sender_email": "alice@nus.edu.sg",
            "event_count": 1,
        },
        {
            "message_id": "m002",
            "status": "Resolved",
            "sender_email": "bob@nus.edu.sg",
            "event_count": 2,
        },
        {
            "message_id": None,
            "status": "Failed",
            "sender_email": "bad-email",
            "event_count": -1,
        },
        {
            "message_id": "m002",
            "status": "Delivered",
            "sender_email": "carol@nus.edu.sg",
            "event_count": 1,
        },
    ]
)

display(df_test)


StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 302425b6-0c7d-428e-9dfe-0fba8c597c30)

## 2. Profile data for AI suggestion

In [12]:
profile_rows = profile_for_dq(df_test, table_name=TABLE_NAME)
display(profile_rows)


StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e22a09eb-d97d-47ac-8435-1ba941cb6158)

## 3. AI suggests candidate DQ rules

In [13]:
responses = suggest_dq_rules(
    profile_rows,
    prompt_template=CONFIG.ai_prompt_config.dq_rule_candidate_template,
    output_col='ai_response',
)
display(responses)


StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1a5dadcb-584a-481d-91f9-8f86f999f203)


## 4. Extract AI candidate rules

Each AI response is still only a **candidate**.
It must go through human approval before it is stored and enforced.


In [14]:
CANDIDATE_DQ_RULES = extract_dq_rules(responses, table_name=TABLE_NAME)
review_dq_rules(CANDIDATE_DQ_RULES, table_name=TABLE_NAME, title='AI suggests, human approves')


StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 16, Finished, Available, Finished, False)

[{'rule_id': 'event_count_value_range',
  'rule_type': 'value_range',
  'columns': ['event_count'],
  'min_value': -1,
  'max_value': 2,
  'severity': 'warning',
  'description': 'Event count should be within the expected range from -1 to 2.'},
 {'rule_id': 'event_count_not_null',
  'rule_type': 'not_null',
  'columns': ['event_count'],
  'severity': 'error',
  'description': 'Event count must be populated and cannot be null.'},
 {'rule_id': 'message_id_not_null',
  'rule_type': 'not_null',
  'columns': ['message_id'],
  'severity': 'error',
  'description': 'MESSAGE_ID must be populated as it uniquely identifies each email message.'},
 {'rule_id': 'message_id_unique_key',
  'rule_type': 'unique_key',
  'columns': ['message_id'],
  'severity': 'error',
  'description': 'MESSAGE_ID must be unique to identify each email message or event record.'},
 {'rule_id': 'sender_email_not_null',
  'rule_type': 'not_null',
  'columns': ['sender_email'],
  'severity': 'error',
  'description': 'Sende


## 5. Human review and approval with widgets

Use the widget below to review AI-suggested rules.
Only approved rules will be stored in the metadata table.

**Important**: Click **Save Approved Rules** before moving to the next step.


In [34]:
APPROVED_DQ_RULES = [rule for rule in CANDIDATE_DQ_RULES if rule.get('approval_status','approved')=='approved']
review_dq_rules(APPROVED_DQ_RULES, table_name=TABLE_NAME, title='Human-approved rules persisted to metadata')


StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 93, Finished, Available, Finished, False)

In [20]:
APPROVED_RULES_FROM_WIDGET

StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 37, Finished, Available, Finished, False)

[{'rule_id': 'event_count_value_range',
  'rule_type': 'value_range',
  'columns': ['event_count'],
  'min_value': 0,
  'max_value': 2,
  'severity': 'warning',
  'description': 'Event count should be within the expected range from -1 to 2.'},
 {'rule_id': 'message_id_not_null',
  'rule_type': 'not_null',
  'columns': ['message_id'],
  'severity': 'error',
  'description': 'MESSAGE_ID must be populated as it uniquely identifies each email message.'},
 {'rule_id': 'message_id_unique_key',
  'rule_type': 'unique_key',
  'columns': ['message_id'],
  'severity': 'error',
  'description': 'MESSAGE_ID must be unique to identify each email message or event record.'},
 {'rule_id': 'sender_email_not_null',
  'rule_type': 'not_null',
  'columns': ['sender_email'],
  'severity': 'error',
  'description': 'Sender email must be populated for every email record.'},
 {'rule_id': 'status_not_null',
  'rule_type': 'not_null',
  'columns': ['status'],
  'severity': 'error',
  'description': 'Status must

## 6. Store approved DQ rules in a metadata table

In [21]:
approved_rules_metadata_df = build_dq_rule_history(
    spark=spark,
    table_name=TABLE_NAME,
    approved_rules=APPROVED_DQ_RULES,
    action_by='notebook_user',
)
approved_rules_metadata_df.write.mode('overwrite').saveAsTable('METADATA_DQ_RULES')
display(approved_rules_metadata_df)


StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bab103e7-9c6e-4aa7-b66e-3394d1c9d433)

## 7. Load approved DQ rules back from the metadata table

In [22]:
APPROVED_DQ_RULES = load_active_dq_rules(
    metadata_df=spark.table('METADATA_DQ_RULES'),
    table_name=TABLE_NAME,
)
review_dq_rules(APPROVED_DQ_RULES, table_name=TABLE_NAME, title='Metadata stores active approved rules')


StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 39, Finished, Available, Finished, False)

[{'rule_id': 'event_count_value_range',
  'rule_type': 'value_range',
  'columns': ['event_count'],
  'min_value': 0,
  'max_value': 2,
  'severity': 'warning',
  'description': 'Event count should be within the expected range from -1 to 2.'},
 {'rule_id': 'message_id_not_null',
  'rule_type': 'not_null',
  'columns': ['message_id'],
  'severity': 'error',
  'description': 'MESSAGE_ID must be populated as it uniquely identifies each email message.'},
 {'rule_id': 'message_id_unique_key',
  'rule_type': 'unique_key',
  'columns': ['message_id'],
  'severity': 'error',
  'description': 'MESSAGE_ID must be unique to identify each email message or event record.'},
 {'rule_id': 'sender_email_not_null',
  'rule_type': 'not_null',
  'columns': ['sender_email'],
  'severity': 'error',
  'description': 'Sender email must be populated for every email record.'},
 {'rule_id': 'status_accepted_values',
  'rule_type': 'accepted_values',
  'columns': ['status'],
  'allowed_values': ['Delivered', 'Res

## 8. Pipeline applies the approved DQ rules loaded from metadata

In [23]:
dq_result = run_dq_rules(df_test, table_name=TABLE_NAME, rules=APPROVED_DQ_RULES)
display(dq_result)


StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c84da470-9d6b-4a25-9c2a-b773ed54e28a)

## 9. Split accepted vs quarantined rows

In [24]:
DQ_RUN_ID = 'run-widget-demo'
df_valid, df_quarantine, df_failure_evidence = split_dq_rows(df_test, APPROVED_DQ_RULES, dq_run_id=DQ_RUN_ID)

display(df_valid)
display(df_quarantine)
display(df_failure_evidence)

# One source row can map to multiple failure-evidence rows via source_row_id + failed_rule_id.
# Quarantine preserves evidence for triage; most true fixes should happen upstream at source.
assert_dq_passed(dq_result)


StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 41, Finished, Available, Finished, False)

DQ run id: 39aa2335-29b9-4491-ba02-54dfdb034e3e
Accepted rows: 1


SynapseWidget(Synapse.DataFrame, 5b3fe3dc-e923-465e-863f-208a8027ba18)

Quarantined source rows: 3


SynapseWidget(Synapse.DataFrame, 50f764ca-69a3-4e50-af6e-b9ca0f589a24)

Failure evidence rows: 5


SynapseWidget(Synapse.DataFrame, d874b46a-a0e4-483c-8008-4c8f7151a9a4)

## Deactivate rules 

In [25]:
ACTIVE_DQ_RULES = load_active_dq_rules(
    metadata_df=spark.table('METADATA_DQ_RULES'),
    table_name=TABLE_NAME,
)
review_dq_rule_deactivations(ACTIVE_DQ_RULES, table_name=TABLE_NAME)


StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 42, Finished, Available, Finished, False)

[{'rule_id': 'event_count_value_range',
  'rule_type': 'value_range',
  'columns': ['event_count'],
  'min_value': 0,
  'max_value': 2,
  'severity': 'warning',
  'description': 'Event count should be within the expected range from -1 to 2.'},
 {'rule_id': 'message_id_not_null',
  'rule_type': 'not_null',
  'columns': ['message_id'],
  'severity': 'error',
  'description': 'MESSAGE_ID must be populated as it uniquely identifies each email message.'},
 {'rule_id': 'message_id_unique_key',
  'rule_type': 'unique_key',
  'columns': ['message_id'],
  'severity': 'error',
  'description': 'MESSAGE_ID must be unique to identify each email message or event record.'},
 {'rule_id': 'sender_email_not_null',
  'rule_type': 'not_null',
  'columns': ['sender_email'],
  'severity': 'error',
  'description': 'Sender email must be populated for every email record.'},
 {'rule_id': 'status_accepted_values',
  'rule_type': 'accepted_values',
  'columns': ['status'],
  'allowed_values': ['Delivered', 'Res

In [33]:
DEACTIVATION_REQUESTS = []  # optional: append {'rule': <rule_dict>, 'action_reason': '...'}
review_dq_rule_deactivations(DEACTIVATION_REQUESTS, table_name=TABLE_NAME, title='Optional deactivation preview')


StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 85, Finished, Available, Finished, False)

StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 86, Finished, Available, Finished, False)

StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 87, Finished, Available, Finished, False)

StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 88, Finished, Available, Finished, False)

StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 89, Finished, Available, Finished, False)

...

StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 98, Finished, Available, Finished, False)

StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 99, Finished, Available, Finished, False)

StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 100, Finished, Available, Finished, False)

StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 101, Finished, Available, Finished, False)

StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 102, Finished, Available, Finished, False)

In [27]:
if DEACTIVATION_REQUESTS:
    deactivation_metadata_df = build_dq_rule_deactivations(
        spark=spark,
        table_name=TABLE_NAME,
        deactivation_requests=DEACTIVATION_REQUESTS,
        action_by='notebook_user',
    )
    deactivation_metadata_df.write.mode('append').saveAsTable('METADATA_DQ_RULES')
    display(deactivation_metadata_df)
else:
    print('No deactivation requests submitted.')


StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 75, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9dbc580c-1255-458d-a4f9-b54a48275ab0)

In [29]:
APPROVED_DQ_RULES = load_latest_active_dq_rules_from_metadata(
    metadata_df=spark.table("METADATA_DQ_RULES"),
    table_name=TABLE_NAME,
)
APPROVED_DQ_RULES

StatementMeta(, e9d2a3ea-6000-452b-926a-677a7aaa68b4, 77, Finished, Available, Finished, False)

<built-in method count of list object at 0x70a2e988c480>


[{'rule_id': 'event_count_value_range',
  'rule_type': 'value_range',
  'columns': ['event_count'],
  'min_value': 0,
  'max_value': 2,
  'severity': 'warning',
  'description': 'Event count should be within the expected range from -1 to 2.'},
 {'rule_id': 'message_id_not_null',
  'rule_type': 'not_null',
  'columns': ['message_id'],
  'severity': 'error',
  'description': 'MESSAGE_ID must be populated as it uniquely identifies each email message.'},
 {'rule_id': 'message_id_unique_key',
  'rule_type': 'unique_key',
  'columns': ['message_id'],
  'severity': 'error',
  'description': 'MESSAGE_ID must be unique to identify each email message or event record.'},
 {'rule_id': 'sender_email_not_null',
  'rule_type': 'not_null',
  'columns': ['sender_email'],
  'severity': 'error',
  'description': 'Sender email must be populated for every email record.'},
 {'rule_id': 'status_not_null',
  'rule_type': 'not_null',
  'columns': ['status'],
  'severity': 'error',
  'description': 'Status must